### Load curated gold layer

In [1]:
import pandas as pd
import os

# Read curated gold layer directly with pandas
df = pd.read_parquet("../data/curated/business_partners")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (20605, 28)
Columns: ['BusinessPartner', 'BusinessPartnerName', 'TotalTransactions', 'TotalAmount', 'AvgAmount', 'StdAmount', 'MaxSingleAmount', 'TotalItems', 'UniqueDocTypes', 'UniqueMainTransactions', 'UniqueContracts', 'ACDocCount', 'ZCDocCount', 'HighValueCount', 'ExtremeValueCount', 'Balance', 'TotalDebit', 'TotalCredit', 'AvgDaysOverdue', 'MaxDaysOverdue', 'AvgDebitCreditRatio', 'ReversalRatio', 'AmountVolatility', 'TotalTransactions_zscore', 'MaxSingleAmount_zscore', 'ReversalRatio_zscore', 'Balance_zscore', 'TotalDebit_zscore']


### Anonymize for portfolio display

In [2]:
import hashlib

def anonymize_name(name):
    if pd.isna(name) or name == "":
        return "CUSTOMER_UNKNOWN"
    hash_val = hashlib.md5(str(name).encode()).hexdigest()[:6].upper()
    return f"CUSTOMER_{hash_val}"

df["BusinessPartnerName"] = df["BusinessPartnerName"].apply(anonymize_name)

### Identify feature columns

In [3]:
feature_cols = [
    "TotalTransactions",
    "TotalAmount",
    "AvgAmount",
    "MaxSingleAmount",
    "UniqueDocTypes",
    "UniqueMainTransactions",
    "UniqueContracts",
    "ACDocCount",
    "ZCDocCount",
    "ReversalRatio",
    "AmountVolatility",
    "Balance",
    "TotalDebit",
    "TotalCredit",
    "AvgDaysOverdue",
    "MaxDaysOverdue",
    "HighValueCount",
    "ExtremeValueCount",
    "TotalTransactions_zscore",
    "MaxSingleAmount_zscore",
    "ReversalRatio_zscore",
    "Balance_zscore",
    "TotalDebit_zscore"
]

X = df[feature_cols].fillna(0)
print("Feature matrix shape:", X.shape)

Feature matrix shape: (20605, 23)


### Train Isolation Forest model

In [4]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)
model.fit(X_scaled)

df["anomaly_flag"]  = model.predict(X_scaled)
df["anomaly_score"] = model.decision_function(X_scaled)
df["risk_score"] = 1 - (
    (df["anomaly_score"] - df["anomaly_score"].min()) /
    (df["anomaly_score"].max() - df["anomaly_score"].min())
)

flagged = df[df["anomaly_flag"] == -1]
print("Total business partners:", len(df))
print("Flagged as anomalous:   ", len(flagged))

Total business partners: 20605
Flagged as anomalous:    1031


### Anomaly watchlist - Top 20 Highest Risk Partners

In [5]:
watchlist = df[df["anomaly_flag"] == -1].sort_values("risk_score", ascending=False)

print("TOP 20 HIGHEST RISK BUSINESS PARTNERS")
print("=" * 60)
print(watchlist[[
    "BusinessPartner", "BusinessPartnerName",
    "TotalTransactions", "MaxSingleAmount",
    "ReversalRatio", "Balance", "risk_score"
]].head(20).to_string(index=False))

TOP 20 HIGHEST RISK BUSINESS PARTNERS
BusinessPartner BusinessPartnerName  TotalTransactions  MaxSingleAmount  ReversalRatio      Balance  risk_score
        1121329     CUSTOMER_DA4EC9               4094      15264558.47      77.484848 -14507853.57    1.000000
        1017842     CUSTOMER_946F8A               2847         79424.45      50.054545   1669780.96    0.989968
        1017947     CUSTOMER_8B86D4               2010         71720.43       0.303349   1363429.03    0.984714
        1008346     CUSTOMER_E6D82A               2016         35726.38       0.174964   1033965.60    0.975301
        1000626     CUSTOMER_A15055               3006        542793.20       0.000000   1663426.53    0.973217
        1120722     CUSTOMER_3063A0               2859         10701.84       0.206384   1212158.96    0.964391
        1015548     CUSTOMER_9DBE0A               1247         72525.93      70.300000    807435.23    0.963356
        1009423     CUSTOMER_3BDDCD               1995         639

### Export watchlist

In [6]:
import os
os.makedirs("../outputs", exist_ok=True)

output = df.sort_values("risk_score", ascending=False)[[
    "BusinessPartner", "BusinessPartnerName",
    "TotalTransactions", "TotalAmount", "AvgAmount",
    "MaxSingleAmount", "ReversalRatio", "Balance",
    "TotalDebit", "AvgDaysOverdue", "anomaly_flag", "risk_score"
]]

output.to_csv("../outputs/anomaly_watchlist.csv", index=False)
print("Watchlist saved to outputs/anomaly_watchlist.csv")
print("Total flagged:", len(output[output["anomaly_flag"] == -1]))

Watchlist saved to outputs/anomaly_watchlist.csv
Total flagged: 1031
